# 🎨 ConPL: Consistent Prototype Learning (ACL 2023 Official Reproduction)
### Few-Shot Continual Relation Extraction on FewRel 5-shot (8 tasks × 10 relations, 6 sequences)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vinhlq2512/research-code-lab/blob/main/notebooks/conpl_fewrel_5shot.ipynb)

Notebook này tái lập kết quả của paper **Consistent Prototype Learning for Few-Shot Continual Relation Extraction (ACL 2023)** trên Google Colab với NVIDIA GPU:
- **Dataset:** FewRel 5-shot (8 tasks, mỗi task 10 relations, 5-shot toàn bộ 8 tasks).
- **Backbone:** `bert-base-uncased` với Prompt-based Tuning (`e1 mask e2 . `).
- **Protocol gốc:** 6 sequences khởi tạo từ `rel_cluster_label_0.npy` với seeds [100, 110, 120, 130, 140, 150].
- **Tiêu chí nghiệm thu (Table 1 Paper):** Cumulative Whole Accuracy tại $T_8 \approx 85.77\%$.


## 1. Kiểm tra GPU Preflight (NVIDIA CUDA Verification)


In [ ]:
import torch

print('=' * 60)
print('KIỂM TRA THIẾT BỊ TÍNH TOÁN (GPU PREFLIGHT)')
print('=' * 60)
if torch.cuda.is_available():
    print(f'✓ CUDA Khả dụng: True')
    print(f'✓ GPU Thiết bị: {torch.cuda.get_device_name(0)}')
    print(f'✓ Bộ nhớ VRAM:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
    !nvidia-smi
else:
    print('⚠️ CẢNH BÁO: CUDA KHÔNG KHẢ DỤNG! Bạn đang chạy trên CPU.')
    print('👉 Hướng dẫn bật GPU: Chọn Runtime trên thanh menu -> Change runtime type -> Chọn T4 GPU -> Lưu.')


## 2. Thiết lập Codebase & Môi trường Dependencies


In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/vinhlq2512/research-code-lab.git'
REPO_NAME = 'research-code-lab'

if not Path(REPO_NAME).exists():
    print(f'Đang clone repository từ {REPO_URL}...')
    !git clone {REPO_URL}

%cd {REPO_NAME}/paper-reproductions/ConPL

# Cài đặt các dependency tương thích với bản transformers nội bộ của ConPL
!pip install -q 'huggingface-hub>=0.8.1,<1.0' 'tokenizers>=0.11.1,<0.14' 'numpy<1.24' scikit-learn wordninja six tqdm


## 3. Kiểm tra Dữ liệu FewRel & Sampler Integrity


In [ ]:
import sys
import json
import numpy as np
sys.path.insert(0, '.')
from encode import BERTSentenceEncoderPrompt
from dataprocess import data_sampler_bert_prompt_deal_first_task

with open('config/config_fewrel_5and10.json', 'r') as f:
    cfg = json.load(f)

cfg['rel_cluster_label'] = 'data/fewrel/CFRLdata_10_100_10_5/rel_cluster_label_0.npy'
cfg['training_file'] = 'data/fewrel/CFRLdata_10_100_10_5/train_0.txt'
cfg['valid_file'] = 'data/fewrel/CFRLdata_10_100_10_5/valid_0.txt'
cfg['test_file'] = 'data/fewrel/CFRLdata_10_100_10_5/test_0.txt'
cfg['first_task_k-way'] = 10
cfg['k-shot'] = 5
cfg['device'] = 'cuda' if torch.cuda.is_available() else 'cpu'

encoder = BERTSentenceEncoderPrompt(cfg)
sampler = data_sampler_bert_prompt_deal_first_task(cfg, encoder.tokenizer, 'e1 mask e2 . ')
sampler.set_seed(100)

print('✓ Số lượng relation:', len(sampler.id2rel) - 1)
print('✓ Data Sampler khởi tạo thành công!')


## 4. Thực thi Huấn luyện ConPL (Full 6 Sequences)


In [ ]:
# Chạy toàn bộ 6 sequences trên GPU với thiết lập chính thức
import os
os.environ['CONPL_DEVICE'] = 'cuda'
os.environ['CONPL_SEQUENCES'] = '6'
os.environ['CONPL_TASKS'] = '8'
os.environ['CONPL_OUTPUT_DIR'] = '../../results/conpl/colab_run_6seq'

!python fewrel_5shot.py 2>&1 | tee ../../results/conpl/colab_run_6seq.log


## 5. Tổng hợp Kết quả & Đối chiếu Paper Table 1


In [ ]:
import json
import pandas as pd
from pathlib import Path

paper_reported = {
    'T1': 95.72, 'T2': 93.53, 'T3': 91.31, 'T4': 89.95,
    'T5': 88.93, 'T6': 88.39, 'T7': 87.43, 'T8': 85.77
}

print('=' * 60)
print('ĐỐI CHIẾU KẾT QUẢ VỚI PAPER CONPL (ACL 2023 TABLE 1)')
print('=' * 60)
for k, v in paper_reported.items():
    print(f'{k}: Paper={v:.2f}%')
